In [ ]:
%load_ext autoreload
%autoreload 2

# Try to enable RAPIDS acceleration (optional, safe)
CUML_AVAILABLE = False
CUDf_AVAILABLE = False
try:
    import cuml
    try:
        cuml.accel.install()
        CUML_AVAILABLE = True
        print("cuml.accel.install() succeeded")
    except Exception as _e:
        print("cuml.accel.install() failed:", _e)   
except Exception as _e:
    print("cuml not available:", _e)

try:
    import cudf.pandas as cudf_pandas
    try:
        cudf_pandas.install()
        CUDf_AVAILABLE = True
        print("cudf.pandas.install() succeeded")
    except Exception as _e:
        print("cudf.pandas.install() failed:", _e)
except Exception as _e:
    print("cudf.pandas not available:", _e)

import pandas as pd
print("pandas version:", pd.__version__)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import LeaveOneGroupOut, HalvingGridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os
import sys
import datetime
from multiprocessing import cpu_count

from pathlib import Path
import sys, os, importlib, importlib.util, traceback

print("cwd:", Path().resolve())
print("os.getcwd():", os.getcwd())
print("\nsys.path (first 12):")
for i,p in enumerate(sys.path[:12]):
    print(i, p)

# 1. Walk cwd parents looking for ml_app (works when notebook is inside the project)
found = None
for p in [Path().resolve()] + list(Path().resolve().parents):
    for name in ("ml_app", "ml-app"):
        if (p / name).is_dir():
            found = p
            print("\nFound project root via parent walk:", found)
            break
    if found:
        break

# 2. Fall back to PROJECT_ROOT env var (set this when launching the container)
if not found:
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root and (Path(env_root) / "ml_app").is_dir():
        found = Path(env_root)
        print("\nFound project root via PROJECT_ROOT env var:", found)

# 3. Fall back to the conventional Docker mount point /workspace/project
if not found:
    candidate = Path("/workspace/project")
    if (candidate / "ml_app").is_dir():
        found = candidate
        print("\nFound project root via /workspace/project mount:", found)

if found:
    # Add project root (for ml_app) and training/ subdir (for training_only)
    for path in (str(found), str(found / "training")):
        if path not in sys.path:
            sys.path.insert(0, path)
    project_root = str(found)
    print("Added to sys.path:", str(found), "and", str(found / "training"))
else:
    print("\nml_app not found — mount the project root into the container:")
    print("  docker run ... -v /host/path/to/project:/workspace/project ...")
    print("  or set the PROJECT_ROOT env var inside the container.")

print("\nimportlib.util.find_spec('ml_app') ->", importlib.util.find_spec("ml_app"))
print("importlib.util.find_spec('training_only') ->", importlib.util.find_spec("training_only"))

try:
    import ml_app
    print("Imported ml_app from:", getattr(ml_app, '__file__', repr(ml_app)))
    from ml_app import WindowingMethod, FeatureExtractor, CustomPipeline
    print("Imported ml_app symbols OK")
except Exception:
    print("ml_app import failed:")
    traceback.print_exc()

try:
    from training_only import SearchableWindowedEstimator
    print("Imported training_only symbols OK")
except Exception:
    print("training_only import failed:")
    traceback.print_exc()

cuml.accel.install() succeeded
cudf.pandas.install() succeeded
pandas version: 2.3.3
cwd: /home/rapids/notebooks
os.getcwd(): /home/rapids/notebooks

sys.path (first 12):
0 /opt/conda/lib/python314.zip
1 /opt/conda/lib/python3.14
2 /opt/conda/lib/python3.14/lib-dynload
3 
4 /opt/conda/lib/python3.14/site-packages

Found project root via /workspace/project mount: /workspace/project
Added to sys.path: /workspace/project and /workspace/project/training

importlib.util.find_spec('ml_app') -> ModuleSpec(name='ml_app', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fe24430b830>, origin='/workspace/project/ml_app/__init__.py', submodule_search_locations=['/workspace/project/ml_app'])
importlib.util.find_spec('training_only') -> ModuleSpec(name='training_only', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fe24430a5d0>, origin='/workspace/project/training/training_only/__init__.py', submodule_search_locations=['/workspace/project/training/training_only'])
I

In [2]:
print(globals().get('project_root', 'NOT SET'))
random_state = 42
n_jobs = 1 if CUML_AVAILABLE and CUDf_AVAILABLE else cpu_count() # // 2  # Use half of the available CPU cores for parallel processing

# project_root is resolved in the previous cell; fall back to relative path for local runs
_project_root = Path(project_root) if 'project_root' in globals() else Path('..').resolve()
data_path = str(_project_root / 'data')
training_path = os.path.join(data_path, 'training', 'converted')
labeled = [p for p in os.listdir(training_path) if p.endswith('.csv') and p.startswith('series_1')]
unlabeled = [p for p in os.listdir(training_path) if p.endswith('.csv') and p.startswith('series_2')]
model_path = os.path.join(data_path, 'models')
training_path, labeled, unlabeled, model_path

/workspace/project


('/workspace/project/data/training/converted',
 ['series_1_sausage_1.csv',
  'series_1_sausage_2.csv',
  'series_1_sausage_3.csv',
  'series_1_sausage_4.csv',
  'series_1_sausage_5.csv'],
 ['series_2_sausage_1.csv',
  'series_2_sausage_2.csv',
  'series_2_sausage_3.csv',
  'series_2_sausage_4.csv',
  'series_2_sausage_5.csv'],
 '/workspace/project/data/models')

In [3]:
sample_data = pd.read_csv(os.path.join(training_path, labeled[0]))
feature_order = sample_data.columns.tolist()
feature_order.remove('label')
feature_order.remove('ws202_light_status')
feature_order.remove('ws202_pir_status')
feature_order

['timestamp',
 'sensor',
 'lse01_conductivity',
 'lse01_moisture',
 'lse01_temperature',
 'ltcfs_temperature']

In [4]:
classifiers = {
    "Logistic Regression": (
        "logreg",
        LogisticRegression(random_state=random_state, ),
        [
            {
                "classifier__solver": ["liblinear"],
                "classifier__penalty": ["l1", "l2"],
                "classifier__C": [0.01, 1, 100],
                "classifier__class_weight": [None, "balanced"],
                "classifier__max_iter": [1000],
            },
            {
                "classifier__solver": ["lbfgs"],
                "classifier__penalty": ["l2"],
                "classifier__C": [0.01, 1, 10],
                "classifier__class_weight": [None, "balanced"],
                "classifier__max_iter": [1000],
            },
            {
                "classifier__solver": ["saga"],
                "classifier__penalty": ["elasticnet"],
                "classifier__l1_ratio": [0.2, 0.5, 0.8],
                "classifier__C": [0.1, 1, 10],
                "classifier__class_weight": ["balanced"],
                "classifier__max_iter": [1000],
            },
        ],
    ),
    "Random Forest": (
        "rf",
        RandomForestClassifier(random_state=random_state),
        {
            "classifier__n_estimators": [200, 500],
            "classifier__max_depth": [None, 10],
            "classifier__min_samples_split": [2],
            "classifier__min_samples_leaf": [1, 4],
            "classifier__max_features": ["sqrt", 0.7],
            "classifier__bootstrap": [True],
            "classifier__class_weight": [None, "balanced_subsample"],
            "classifier__criterion": ["gini"],
        },
    ),
    "MLP Classifier": (
        "mlp",
        MLPClassifier(random_state=random_state),
        {
            "classifier__hidden_layer_sizes": [
                (32),
                (32, 16),
                (64, 32),
                (64, 32, 16),
            ],
            "classifier__activation": ["relu", "tanh"],
            "classifier__alpha": [1e-4, 1e-3],
            "classifier__learning_rate_init": [3e-4, 1e-3],
            "classifier__batch_size": [32],
            "classifier__max_iter": [500],
            "classifier__early_stopping": [True],
            "classifier__validation_fraction": [0.15],
        },
    ),
}

In [5]:
def create_pipeline(classifier_name: str) -> CustomPipeline:
    if classifier_name not in classifiers:
        raise ValueError(
            f"Classifier '{classifier_name}' not found. Available classifiers: {list(classifiers.keys())}"
        )

    classifier_key, classifier, _ = classifiers[classifier_name]

    pipeline = CustomPipeline(
        [
            (
                "windowing",
                WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2),
            ),
            (
               "inputer",
               SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier", 
                classifier
            ),
        ],
        version=f"{classifier_key}_{datetime.datetime.now().strftime('%Y%m%d')}_v1",
        model_type=classifier_name,
        feature_order=feature_order,
    )

    return pipeline

In [6]:
pipelines = {
    name: create_pipeline(name) for name in classifiers.keys()
}

In [7]:
for name, pipeline in pipelines.items():
    print(f"Pipeline for {name}:")
    pipeline.display()
    print("\n" + "="*50 + "\n")

Pipeline for Logistic Regression:
Pipeline Version: logreg_20260527_v1
Model Type: Logistic Regression
Feature Order: ['timestamp', 'sensor', 'lse01_conductivity', 'lse01_moisture', 'lse01_temperature', 'ltcfs_temperature']
Created At: 2026-05-27T20:10:14.039182
Pipeline Steps:
 - windowing: WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2)
 - inputer: SimpleImputer(strategy='median')
 - scaler: StandardScaler()
 - classifier: LogisticRegression(random_state=42)


Pipeline for Random Forest:
Pipeline Version: rf_20260527_v1
Model Type: Random Forest
Feature Order: ['timestamp', 'sensor', 'lse01_conductivity', 'lse01_moisture', 'lse01_temperature', 'ltcfs_temperature']
Created At: 2026-05-27T20:10:14.039182
Pipeline Steps:
 - windowing: WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2)
 - inputer: SimpleImputer(strategy='median')
 - scaler: StandardScaler()
 - classifier: RandomForestClassifier(random_state=42)


Pipeline for MLP Classifier:
Pipelin

In [8]:
labeled_data = []
for file in labeled:
    data = pd.read_csv(os.path.join(training_path, file))
    data['group'] = file.split('.')[0]
    labeled_data.append(data)
labeled_df = pd.concat(labeled_data, ignore_index=True)
labeled_df['timestamp'] = pd.to_datetime(labeled_df['timestamp'], utc=True, format="ISO8601")
labeled_df.sort_values(by='timestamp', inplace=True)
labeled_df.head(), labeled_df.shape

(                              timestamp sensor  lse01_conductivity  \
 259506 2023-03-30 00:01:27.651646+00:00  lse01              5930.0   
 0      2023-03-30 00:01:39.515132+00:00  lse01              3851.0   
 194627 2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 129753 2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 129754 2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 
         lse01_moisture  lse01_temperature  ltcfs_temperature  \
 259506            6.93               9.78                NaN   
 0                 1.46               9.88                NaN   
 194627             NaN                NaN                NaN   
 129753             NaN                NaN                NaN   
 129754             NaN                NaN                NaN   
 
         ws202_light_status  ws202_pir_status      label               group  
 259506                 NaN               NaN  Umroetung  series_1_sausage_5  
 0                    

In [9]:
group_count = labeled_df['group'].nunique()
avg_rows_per_group = int(np.round(np.mean(labeled_df.groupby('group').size())))
group_cv = LeaveOneGroupOut()
splits = list(group_cv.split(labeled_df[feature_order], labeled_df['label'], groups=labeled_df['group']))
splits, group_count, avg_rows_per_group

([(array([     0,      2,      3, ..., 324373, 324375, 324376],
         shape=(259509,)),
   array([     1,     25,     26, ..., 324364, 324368, 324374],
         shape=(64868,))),
  (array([     0,      1,      2, ..., 324374, 324375, 324376],
         shape=(259492,)),
   array([    14,     15,     16, ..., 324346, 324366, 324370],
         shape=(64885,))),
  (array([     0,      1,      2, ..., 324373, 324374, 324376],
         shape=(259503,)),
   array([     3,      4,      5, ..., 324365, 324371, 324375],
         shape=(64874,))),
  (array([     0,      1,      3, ..., 324372, 324374, 324375],
         shape=(259498,)),
   array([     2,     20,     21, ..., 324367, 324373, 324376],
         shape=(64879,))),
  (array([     1,      2,      3, ..., 324374, 324375, 324376],
         shape=(259506,)),
   array([     0,      9,     10, ..., 324362, 324369, 324372],
         shape=(64871,)))],
 5,
 64875)

In [10]:
hyperparameter_selections = {}
hpo_scores = {}
for name, (key, classifier, param_grid) in classifiers.items():
    if isinstance(param_grid, dict):
        keys = param_grid.keys()
    elif isinstance(param_grid, list) and len(param_grid) > 0:
        keys_set = set()
        for grid in param_grid:
            if isinstance(grid, dict):
                keys_set.update(grid.keys())
        keys = list(keys_set)
    else:
        raise ValueError(f"Unexpected type for param_grid: {type(param_grid)}")
    hyperparameter_selections[name] = {k: None for k in keys}
    hpo_scores[name] =  np.inf

hyperparameter_selections

{'Logistic Regression': {'classifier__class_weight': None,
  'classifier__penalty': None,
  'classifier__l1_ratio': None,
  'classifier__solver': None,
  'classifier__C': None,
  'classifier__max_iter': None},
 'Random Forest': {'classifier__n_estimators': None,
  'classifier__max_depth': None,
  'classifier__min_samples_split': None,
  'classifier__min_samples_leaf': None,
  'classifier__max_features': None,
  'classifier__bootstrap': None,
  'classifier__class_weight': None,
  'classifier__criterion': None},
 'MLP Classifier': {'classifier__hidden_layer_sizes': None,
  'classifier__activation': None,
  'classifier__alpha': None,
  'classifier__learning_rate_init': None,
  'classifier__batch_size': None,
  'classifier__max_iter': None,
  'classifier__early_stopping': None,
  'classifier__validation_fraction': None}}

In [11]:
# ── HPO with pre-computed windowed features ─────────────────────────────────
#
# WHY: WindowingMethod.transform() is a pure-Python loop over every window ×
# every sensor × every feature column.  With 324k raw rows and 5-fold LOGO-CV,
# re-running it inside every GridSearchCV fit costs ~5 min/fold.  The windowed
# feature matrix is identical for all hyperparameter candidates (window_size
# and stride are fixed), so we compute it exactly once per CV split and hand
# the pre-windowed arrays to a lightweight imputer→scaler→classifier Pipeline.
#
# The production CustomPipeline (including the windowing step) is NOT changed –
# best params are applied to it in the final evaluation cell.

import time
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone as sklearn_clone

WINDOW_SIZE = 10
STRIDE = 2

# ── 1. Pre-compute windowed features for every group ─────────────────────────
print("Pre-computing windowed features for all groups...")
t0 = time.time()

_wm = WindowingMethod(method=FeatureExtractor(), window_size=WINDOW_SIZE, stride=STRIDE)
_le = LabelEncoder()

_group_windows = {}   # group_name → (X_win: DataFrame, y_win: ndarray)
for grp, grp_df in labeled_df.groupby('group', sort=False):
    grp_df = grp_df.sort_values('timestamp').reset_index(drop=True)
    X_grp  = grp_df[feature_order].reset_index(drop=True)
    y_grp  = grp_df['label'].reset_index(drop=True)

    X_win = _wm.transform(X_grp)   # <-- the expensive step, done once per group

    n_rows = len(X_grp)
    starts = list(range(0, n_rows - WINDOW_SIZE + 1, STRIDE))
    labels = [y_grp.iloc[s : s + WINDOW_SIZE].mode().iloc[0] for s in starts]

    _group_windows[grp] = (X_win, np.array(labels))

# Combine into one flat dataset + per-window group membership array
_all_X   = pd.concat([v[0] for v in _group_windows.values()], ignore_index=True)
_all_y   = np.concatenate([v[1] for v in _group_windows.values()])
_all_grp = np.concatenate([
    np.full(len(v[0]), g) for g, v in _group_windows.items()
])
_all_y_enc = _le.fit_transform(_all_y)

print(f"  Windowed dataset: {_all_X.shape[0]:,} windows × {_all_X.shape[1]} features  "
      f"({time.time()-t0:.1f}s)")

# ── 2. Run HPO on the pre-windowed data ──────────────────────────────────────
hyperparameter_selections = {}
hpo_scores = {}

for name, (key, clf_proto, param_grid) in classifiers.items():
    print(f"\nTuning hyperparameters for {name}...")

    all_keys = list(param_grid.keys()) if isinstance(param_grid, dict) \
               else list({k for g in param_grid for k in g})
    hyperparameter_selections[name] = {k: None for k in all_keys}

    # Plain sklearn Pipeline — no windowing step, param keys are unchanged
    hpo_pipeline = SklearnPipeline([
        ("inputer",    SimpleImputer(strategy="median")),
        ("scaler",     StandardScaler()),
        ("classifier", sklearn_clone(clf_proto)),
    ])

    search = HalvingGridSearchCV(
        estimator=hpo_pipeline,
        param_grid=param_grid,          # original "classifier__*" keys, no prefix change
        factor=4,
        resource='n_samples',
        min_resources=len(_all_X) // (group_count * 10),
        random_state=random_state,
        cv=LeaveOneGroupOut(),
        verbose=3,
        n_jobs=n_jobs,
        error_score='raise',
    )

    t1 = time.time()
    search.fit(_all_X, _all_y_enc, groups=_all_grp)
    print(f"  Search completed in {(time.time()-t1)/60:.1f} min")

    best_params = search.best_params_
    for param, value in best_params.items():
        hyperparameter_selections[name][param] = value

    hpo_scores[name] = search.best_score_
    print(f"  Best score : {search.best_score_:.4f}")
    print(f"  Best params: {best_params}")

hyperparameter_selections, hpo_scores

Pre-computing windowed features for all groups...


/opt/conda/lib/python3.14/site-packages/cudf/pandas/_wrappers/numpy.py:221: RuntimeWarning: Mean of empty slice
  func(*slow_args, **slow_kwargs), func, *args, **kwargs
/opt/conda/lib/python3.14/site-packages/cupy/_statistics/order.py:115: RuntimeWarning: All-NaN slice encountered
  warnings.warn('All-NaN slice encountered', RuntimeWarning)
/opt/conda/lib/python3.14/site-packages/cupy/_statistics/order.py:146: RuntimeWarning: All-NaN slice encountered
  warnings.warn('All-NaN slice encountered', RuntimeWarning)
/opt/conda/lib/python3.14/site-packages/cudf/pandas/_wrappers/numpy.py:221: RuntimeWarning: Mean of empty slice
  func(*slow_args, **slow_kwargs), func, *args, **kwargs
/opt/conda/lib/python3.14/site-packages/cupy/_statistics/order.py:115: RuntimeWarning: All-NaN slice encountered
  warnings.warn('All-NaN slice encountered', RuntimeWarning)
/opt/conda/lib/python3.14/site-packages/cupy/_statistics/order.py:146: RuntimeWarning: All-NaN slice encountered
  warnings.warn('All-NaN sl

  Windowed dataset: 162,167 windows × 29 features  (5119.8s)

Tuning hyperparameters for Logistic Regression...
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 3243
max_resources_: 162167
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 27
n_resources: 3243
Fitting 5 folds for each of 27 candidates, totalling 135 fits


/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope' 'ltcfs_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope' 'ltcfs_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope' 'ltcfs_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[CV 1/5] END classifier__C=0.01, classifier__class_weight=None, classifier__max_iter=1000, classifier__penalty=l1, classifier__solver=liblinear;, score=(train=0.734, test=0.715) total time=  13.4s
[CV 2/5] END classifier__C=0.01, classifier__class_weight=None, classifier__max_iter=1000, classifier__penalty=l1, classifier__solver=liblinear;, score=(train=0.715, test=0.690) total time=   1.8s


/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: 

... (784 outputs left out)

[CV 5/5] END classifier__activation=tanh, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32), classifier__learning_rate_init=0.0003, classifier__max_iter=500, classifier__validation_fraction=0.15;, score=(train=0.903, test=0.654) total time=   9.3s
[CV 1/5] END classifier__activation=relu, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.001, classifier__max_iter=500, classifier__validation_fraction=0.15;, score=(train=0.916, test=0.630) total time=   5.2s
[CV 2/5] END classifier__activation=relu, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.001, classifier__max_iter=500, classifier__validation_fraction=0.15;, score=(train=0.931, test=0.732) total time=  10.2s
[CV 3/5] END classifie

/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/conda/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['lse01_conductivity_slope' 'lse01_moisture_slope'
 'lse01_temperature_slope']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[CV 5/5] END classifier__activation=relu, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.001, classifier__max_iter=500, classifier__validation_fraction=0.15;, score=(train=0.924, test=0.703) total time=   7.3s
----------
iter: 2
n_candidates: 2
n_resources: 51888
Fitting 5 folds for each of 2 candidates, totalling 10 fits
[CV 1/5] END classifier__activation=relu, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.001, classifier__max_iter=500, classifier__validation_fraction=0.15;, score=(train=0.931, test=0.631) total time=  27.0s
[CV 2/5] END classifier__activation=relu, classifier__alpha=0.0001, classifier__batch_size=32, classifier__early_stopping=True, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.001, classifier__max_iter=

({'Logistic Regression': {'classifier__class_weight': None,
   'classifier__penalty': 'l2',
   'classifier__l1_ratio': None,
   'classifier__solver': 'liblinear',
   'classifier__C': 100,
   'classifier__max_iter': 1000},
  'Random Forest': {'classifier__n_estimators': 500,
   'classifier__max_depth': None,
   'classifier__min_samples_split': 2,
   'classifier__min_samples_leaf': 4,
   'classifier__max_features': 'sqrt',
   'classifier__bootstrap': True,
   'classifier__class_weight': 'balanced_subsample',
   'classifier__criterion': 'gini'},
  'MLP Classifier': {'classifier__hidden_layer_sizes': (64, 32, 16),
   'classifier__activation': 'relu',
   'classifier__alpha': 0.0001,
   'classifier__learning_rate_init': 0.001,
   'classifier__batch_size': 32,
   'classifier__max_iter': 500,
   'classifier__early_stopping': True,
   'classifier__validation_fraction': 0.15}},
 {'Logistic Regression': np.float64(0.751047556776854),
  'Random Forest': np.float64(0.7426857399873243),
  'MLP Class

In [ ]:
# X_train = labeled_df[feature_order].copy()
# X_train['__groups__'] = labeled_df['group']
# y_train = labeled_df['label']
# groups_train = labeled_df['group']

# for name, pipeline in pipelines.items():
#     print(f"Tuning hyperparameters for {name}...")

#     cv = LeaveOneGroupOut()

#     # wrap pipeline to perform windowing per-group (expands X/y into windows)
#     # Use the training-only clone-safe wrapper so GridSearchCV can clone the estimator
#     estimator = SearchableWindowedEstimator(
#         pipeline=pipeline, window_size=10, stride=2
#     )

#     # classifiers may provide param grids as dict or list-of-dicts; prefix keys so they target the inner pipeline
#     orig_param_grid = classifiers[name][2]

#     def _prefix_grid(grid):
#         if isinstance(grid, dict):
#             return {f"pipeline__{k}": v for k, v in grid.items()}
#         elif isinstance(grid, list):
#             return [{f"pipeline__{k}": v for k, v in g.items()} for g in grid]
#         else:
#             raise ValueError(f"Unexpected type for param_grid: {type(grid)}")

#     param_grid_prefixed = _prefix_grid(orig_param_grid)

#     search = HalvingGridSearchCV(
#         estimator=estimator,
#         param_grid=param_grid_prefixed,
#         factor=4,
#         resource='n_samples',
#         min_resources=avg_rows_per_group // 10,
#         random_state=random_state,
#         cv=cv,
#         verbose=3,
#         n_jobs=n_jobs,
#         error_score='raise'
#     )

#     print(f"  Starting hyperparameter search for {name}...")a
#     search.fit(X_train, y_train, groups=groups_train)
#     print(f"  Hyperparameter search completed for {name}.")

#     best_params = search.best_params_
#     for param, value in best_params.items():
#         param_name = param.replace(
#             "pipeline__", ""
#         )  # Remove prefix to store original param name
#         hyperparameter_selections[name][param_name] = value

#     hpo_scores[name] = search.best_score_
#     print(f"  Best hyperparameters: {best_params}")

# hyperparameter_selections, hpo_scores

In [ ]:
best_hyperparameters = {name: {} for name in classifiers.keys()}
for name, params in hyperparameter_selections.items():
    best_scores = hpo_scores[name]
    for param, values in params.items():
        vals = {}
        for val in values:
            if val in vals.keys():
                continue
            scores = [best_scores[i][1] for i in range(len(values)) if values[i] == val]
            vals[val] = np.sum(scores)
        best_val = max(vals, key=vals.get)
        best_hyperparameters[name][param] = best_val

best_hyperparameters

In [ ]:
scores = {k: [] for k in classifiers.keys()}

for i, (train_idx, test_idx) in enumerate(splits):
    print(f"Processing fold {i+1}/{len(splits)}...")
    X_train = labeled_df.iloc[train_idx][feature_order]
    y_train = labeled_df.iloc[train_idx]['label']
    X_test  = labeled_df.iloc[test_idx][feature_order]
    y_test  = labeled_df.iloc[test_idx]['label']
    groups_train = labeled_df.iloc[train_idx]['group']

    if hasattr(X_train, "to_pandas") and n_jobs > 1:
        X_train = X_train.to_pandas()
        y_train = y_train.to_pandas()
        X_test = X_test.to_pandas()
        y_test = y_test.to_pandas()
        groups_train = groups_train.to_pandas()
    
    for name, pipeline in pipelines.items():
        print(f"  Evaluating {name} on fold {i+1}...")
        pipeline.set_params(**best_hyperparameters[name])
        pipeline.fit(X_train, y_train)
        score = pipeline.score(X_test, y_test)
        proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
        scores[name].append((score, proba))

for name, fold_scores in scores.items():
    fold_accuracies = [s[0] for s in fold_scores]
    avg_accuracy = np.mean(fold_accuracies)
    fold_probas = [s[1] for s in fold_scores]
    avg_proba = np.mean(fold_probas, axis=0) if fold_probas[0] is not None else None
    print(f"{name} - Average Accuracy across folds: {avg_accuracy:.4f} - Average Proba across folds: {avg_proba}")